# 강의 05 · 실습 2 — 평가 데이터셋과 채점자 · (4) 고난도 I


## 1. 문제상황

- 놀이공원 구름월드의 안내 서비스에는 현재 버전 A가 붙어 있습니다. 버전 A는 두 문장 안에서 답만 합니다.
- 운영팀은 답 앞뒤에 감사 인사와 마무리 문장을 붙이는 새 버전 B를 만들었고, 어느 쪽을 배포할지 정해야 합니다.
- 운영팀이 지금까지 쓰던 채점 기준은 두 가지입니다. 답이 정답과 부합하는가, 답이 정중한가입니다.
- 두 기준으로 채점하면 버전 B가 앞섭니다. 그런데 이용자 설문에서 「답이 길어서 읽기 불편하다」는 응답이 늘었습니다.
- 운영팀은 간결성을 세 번째 기준으로 추가하려 합니다. 기준 하나를 추가했을 때 순위가 어떻게 되는지 아무도 계산해 보지 않았습니다.


## 2. 문제와 목표

- **문제**: 배포 결정이 채점 기준 두 개에 묶여 있고, 기준을 하나 추가했을 때 순위가 바뀌는지를 사람이 계산해 본 적이 없습니다.
- **목표**
  - 같은 골든 데이터셋으로 버전 A와 버전 B를 각각 평가합니다.
    - 두 버전: A는 두 문장 안에서 답만 하고, B는 답 앞뒤에 감사 인사와 마무리 문장을 붙입니다(지시문은 단계 0의 서비스 코드에 있습니다)
    - 데이터셋 `sesac-lec05-ex02-compare`: 질문 6개(FAQ 안 4개 — 인사 1개 포함, 밖 2개)와 정답 텍스트의 짝. 값(`GOLDEN`)과 모델 채점자에 넣는 채점 지시문(`JUDGE_GUIDE`)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
  - 채점자 세 개의 점수를 버전마다 모읍니다.
    - 정답 부합 `judge_faithful`: 모델 채점, 덧붙은 인사말·마무리 문장은 어긋남으로 보지 않음(지시문은 단계 0에 주어져 있습니다)
    - 정중함 `rule_polite`: 답에 「말씀해 주세요」가 있으면 1.0, 없고 존댓말이면 0.5, 둘 다 아니면 0
    - 간결성 `rule_concise`: 마침표·느낌표·물음표로 나눈 문장 수가 2 이하이면 1, 3 이상이면 0
  - 기준 두 개일 때의 순위와 기준 세 개일 때의 순위를 나란히 출력하고, 기준 세 개 총점의 1위를 배포 후보로 정합니다.
    - 기준 두 개: 정답 부합·정중함 / 기준 세 개: 정답 부합·정중함·간결성
- **목표 달성 여부의 판정 기준**:
  - 버전마다 채점자 세 개의 평균이 표로 출력되고,
  - 기준 두 개(정답 부합·정중함) 총점에서는 버전 B가 1위, 기준 세 개 총점에서는 버전 A가 1위로 출력되는 것을 확인합니다.
  - 마지막 줄에 배포 후보 A가 출력됩니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex02_s4_diagram.svg)


## 4. 단계별 요구사항

1. **골든 데이터셋을 등록합니다.**
    - 질문 6개(FAQ 안 4개 — 인사 1개 포함, FAQ 밖 2개)와 정답 텍스트(`answer`)·FAQ 안 질문 여부(`must_know`)를 짝지어 `sesac-lec05-ex02-compare` 데이터셋으로 올립니다.
    - 같은 이름의 데이터셋이 이미 있으면 재사용합니다.
    - 질문과 정답 텍스트는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
2. **평가 대상을 두 개 지정합니다.**
    - `target_a`는 버전 A 서비스 `answer_a`를, `target_b`는 버전 B 서비스 `answer_b`를 불러 `{"answer": 답}`을 돌려줍니다.
3. **채점자 두 개를 만듭니다.**
    - `judge_faithful`은 `Judge` 스키마를 강제한 모델에 질문·정답·답을 넘겨 정답 부합을 판정합니다.
    - `rule_polite`는 답에 「말씀해 주세요」가 있으면 1.0, 없고 존댓말(「습니다」「세요」)이면 0.5, 둘 다 아니면 0을 냅니다.
    - `judge_faithful`의 지시문은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
4. **채점자 하나를 추가합니다.**
    - `rule_concise`는 답을 마침표·느낌표·물음표로 나눈 문장 수가 2 이하이면 1, 3 이상이면 0을 냅니다.
5. **실험을 버전마다 실행합니다.**
    - `run_version` 함수가 버전 이름과 평가 대상을 받아 `client.evaluate`를 실행하고, 채점자별 평균을 딕셔너리로 돌려줍니다.
    - 실험 URL 줄은 `redirect_stdout`으로 잡고 「실험 URL: (LangSmith 화면에서 확인)」을 대신 출력합니다.
    - 실험 이름 접두어는 `compare`이고 그 뒤에 버전 이름(a·b)을 붙입니다.
6. **비교 표와 배포 후보를 출력합니다.**
    - 버전별 채점자 평균 표, 기준 두 개(`judge_faithful`·`rule_polite`) 총점과 순위, 기준 세 개 총점과 순위를 차례로 출력하고, 기준 세 개 총점의 1위를 「배포 후보」로 출력합니다.
    - 마지막 줄은 「배포 후보(기준 3개 총점 1위): 버전 X」 형식입니다.


## 5. 코드 골격 — LangSmith 평가 4단

골격은 네 단계입니다. 평가 대상이 둘, 채점자가 셋, 실행이 두 번이라는 점만 다릅니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 데이터셋 등록 | 질문 6개와 정답 텍스트를 짝지어 데이터셋으로 올립니다 | `client.create_dataset(...)`, `client.create_examples(...)` | 1 |
| ② 평가 대상 지정 | 버전 A·B를 감싸는 함수 두 개를 평가 대상으로 지정합니다 | `def target_a(inputs)`, `def target_b(inputs)` | 2 |
| ③ evaluator 정의 | 채점 함수 세 개를 만듭니다. 모델 채점 하나, 규칙 채점 둘입니다 | `with_structured_output(Judge)`, `def rule_polite(...)`, `def rule_concise(...)` | 3, 4 |
| ④ 실행·순위 | 버전마다 실험을 돌리고, 총점 순위 두 벌을 매겨 배포 후보를 고릅니다 | `client.evaluate(...)` 2회, 순위 계산 | 5, 6 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델을 준비합니다. `warnings.filterwarnings` 두 줄은 추적 라이브러리가 내는 직렬화 경고와 진행 막대 경고를 화면에서 감춥니다. 동작에는 영향이 없습니다. `re`는 답의 문장 수를 세는 규칙 채점자에서 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- `LANGSMITH_TRACING`을 켜고 프로젝트 이름을 `sesac-lec05-ex02`로 정하면, 아래에서 부르는 모델 호출과 실험이 LangSmith의 그 프로젝트에 남습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import io   # io — io.StringIO()에 client.evaluate가 출력하는 실험 URL 줄을 잡아 둡니다
import os
import re
import warnings
from contextlib import redirect_stdout

from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langsmith import Client
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", message="IProgress not found")

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex02"

# 이 실습에서 부르는 모델 이름
MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
llm = init_chat_model(MODEL, model_provider="litellm")
print("모델 준비를 마쳤습니다. LangSmith 프로젝트:", os.environ["LANGSMITH_PROJECT"])

# 주어진 자료: 골든 데이터셋 GOLDEN(질문·정답 짝)과 모델 채점자에 넣는 지시문 JUDGE_GUIDE — 값을 그대로 씁니다
GOLDEN = [
    {"question": "안녕하세요!",
     "answer": "안녕하세요! 무엇을 도와드릴까요?", "must_know": True},
    {"question": "자유이용권 환불 규정 알려 주세요",
     "answer": "이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.", "must_know": True},
    {"question": "야간개장 때 퍼레이드 하나요?",
     "answer": "야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.", "must_know": True},
    {"question": "주차 요금은 얼마인가요?",
     "answer": "자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.", "must_know": True},
    {"question": "근처에 맛있는 식당이 있나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
    {"question": "파이썬 리스트 정렬은 어떻게 하나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
]

JUDGE_GUIDE = (
    "너는 엄격한 채점자다. 평가 대상의 답이 정답(reference)과 부합하는지, 어긋난 지점이 없는지 판정하라. "
    "인사말이나 마무리 문장이 덧붙은 것, FAQ에 적힌 관련 안내가 덧붙은 것은 어긋남으로 보지 않는다. "
    "정답에 없는 사실을 지어냈거나 정답과 다른 내용을 말했으면 어긋남이다."
)


평가 대상이 될 안내 서비스 두 버전입니다. 두 버전 모두 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_service`가 지시문을 받아 서비스 함수를 만들고, 버전 A와 버전 B는 지시문만 다릅니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

BASE_GUIDE = (
    "너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
    "인사말에는 짧은 인사로 답한다. "
    "FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다."
)


def make_service(guide: str):
    """안내 서비스를 만든다: 지시문 하나를 받아 질문 -> 답 함수를 돌려준다."""
    system = guide + "\n=== FAQ ===\n" + FAQ

    def answer(question: str) -> str:
        res = llm.invoke([("system", system), ("human", question)])
        return res.content.strip()

    return answer


GUIDE_A = BASE_GUIDE
GUIDE_B = (
    BASE_GUIDE
    + " 답의 앞에 '문의해 주셔서 감사합니다.'를 붙이고, 답의 뒤에 '더 궁금한 점이 있으면 말씀해 주세요.'를 붙인다."
)

answer_a = make_service(GUIDE_A)
answer_b = make_service(GUIDE_B)

print("[A]", answer_a("주차 요금은 얼마인가요?"))
print("[B]", answer_b("주차 요금은 얼마인가요?"))

### 단계 ① — 데이터셋 등록 (요구사항 1)

질문 6개와 정답 텍스트를 짝지어 올립니다. 정답 텍스트는 FAQ의 답 문장 그대로이고, FAQ 밖 질문 2개의 정답은 「해당 내용은 확인할 수 없습니다.」입니다.


In [ ]:
# 여기에 단계 ①을 작성합니다.

### 단계 ② — 평가 대상 지정 (요구사항 2)

평가 대상은 버전마다 하나씩 두 개입니다. 둘 다 `inputs`에서 질문을 꺼내 자기 버전의 서비스를 부르고 `{"answer": 답}`을 돌려줍니다.


In [ ]:
# 여기에 단계 ②을 작성합니다.

### 단계 ③ — evaluator 정의 (요구사항 3, 4)

- 채점자는 `inputs`·`outputs`·`reference_outputs` 세 딕셔너리를 받아 `{"key": 이름, "score": 점수}` 딕셔너리를 돌려주는 함수입니다.
- `judge_faithful`은 답을 정답 텍스트와 비교하는 정답 부합 채점자입니다. 지시문에 「인사말·마무리 문장·FAQ에 적힌 관련 안내가 덧붙은 것은 어긋남이 아니다」를 적어, 길이 차이가 정답 부합 점수에 섞이지 않게 합니다.
- `rule_polite`는 0·0.5·1.0 세 값을 냅니다. 점수가 0과 1 사이 실수여도 LangSmith는 그대로 기록합니다.
- `rule_concise`는 문장 수만 봅니다. 문장 수는 마침표·느낌표·물음표로 나눈 조각 가운데 비어 있지 않은 조각의 수입니다.


In [ ]:
# 여기에 단계 ③을 작성합니다.

### 단계 ④ — 실행·게이트 (요구사항 5, 6)

`run_version`이 버전 하나를 평가하고 채점자별 평균을 돌려줍니다. 두 버전을 돌린 뒤, 기준 두 개 총점과 기준 세 개 총점을 각각 계산해 순위를 매깁니다. 게이트는 기준 세 개 총점의 1위를 배포 후보로 고릅니다.

`client.evaluate(…, experiment_prefix=접두어)`의 결과 한 건 `r`에서 질문은 `r["example"].inputs["question"]`, 채점 결과 목록은 `r["evaluation_results"]["results"]`(원소마다 `key`·`score`)입니다.


In [ ]:
# 여기에 단계 ④을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 0의 서비스 출력에서 버전 A의 답은 두 문장 이하이고, 버전 B의 답은 「문의해 주셔서 감사합니다.」로 시작해 「더 궁금한 점이 있으면 말씀해 주세요.」로 끝납니다.
2. 단계 ④ 표에서 두 버전의 `judge_faithful` 평균이 같거나 비슷하고, `rule_polite`는 B가 높고, `rule_concise`는 A가 높습니다.
3. 기준 두 개 총점의 순위는 B > A이고, 기준 세 개 총점의 순위는 A > B입니다.
4. 마지막 줄에 「배포 후보(기준 3개 총점 1위): 버전 A」가 출력됩니다.

네 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.
